In [45]:
import torch
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration
from translation_dataset import create_dataloader, TranslationDataset
from prefix_t5_model import PrefixT5
from trainer import Trainer, TrainingHyperParams, load_model
from prediction import generate_translation
from configs import CHECKPOINT, DATASET, BATCH_SIZE

In [46]:
t5_tokenizer = T5Tokenizer.from_pretrained(CHECKPOINT)
t5_model = T5ForConditionalGeneration.from_pretrained(CHECKPOINT)

## Data Preprocessing

In [3]:
# Load dataset
dataset = load_dataset(DATASET, "en-fr")['train']

# Shuffle the dataset
dataset = dataset.shuffle(seed=42)

# Split the dataset into train, valid, test (train_size=80%, valid_size=10%, test_size=10%)
train_valid_dataset, test_dataset = torch.utils.data.random_split(dataset, [0.9, 0.1])
train_dataset, valid_dataset = torch.utils.data.random_split(train_valid_dataset, [0.9, 0.1])

In [4]:
train_translation_dataset = TranslationDataset(
    data=train_dataset,
    tokenizer=t5_tokenizer,
    shift_right_func=t5_model._shift_right
)

valid_translation_dataset = TranslationDataset(
    data=valid_dataset,
    tokenizer=t5_tokenizer,
    shift_right_func=t5_model._shift_right
)

test_translation_dataset = TranslationDataset(
    data=test_dataset,
    tokenizer=t5_tokenizer,
    shift_right_func=t5_model._shift_right
)


In [5]:
train_loader = create_dataloader(
    dataset=train_translation_dataset, 
    batch_size=BATCH_SIZE
)

valid_loader = create_dataloader(
    dataset=valid_translation_dataset, 
    batch_size=BATCH_SIZE
)

test_loader = create_dataloader(
    dataset=test_translation_dataset, 
    batch_size=BATCH_SIZE, 
)

In [8]:
print(f"Number of batches in train: {len(train_loader)}")
print(f"Number of batches in valid: {len(valid_loader)}")
print(f"Number of batches in test: {len(test_loader)}")

Number of batches in train: 6434
Number of batches in valid: 715
Number of batches in test: 795


In [9]:
# Get a sample batch
sample_batch = next(iter(train_loader))
    
print("\nSample batch shapes:")
for key, value in sample_batch.items():
    print(f"{key}: {value.shape}")

# Decode a sample from the batch
print("\n")
print("Input:", t5_tokenizer.decode(sample_batch['input_ids'][0].tolist(), skip_special_tokens=True))
print("Target:", t5_tokenizer.decode(sample_batch['labels'][0].tolist(), skip_special_tokens=True))


Sample batch shapes:
input_ids: torch.Size([16, 128])
attention_mask: torch.Size([16, 128])
labels: torch.Size([16, 128])
decoder_input_ids: torch.Size([16, 128])


Input: translate English to French: And yet, in accord with theories she believed right, she wanted to make herself in love with him.
Target: Cependant, d’après des théories qu’elle croyait bonnes, elle voulut se donner de l’amour.


## Prefix T5 Model

In [ ]:
prefixed_t5_model = PrefixT5(
    t5_model=t5_model, prefix_length=10, prefix_projection_size=512
)

## Training

In [11]:
sample_data = dataset.select(range(50))
sample_valid = dataset.select(range(50, 70))
sample_trans_data = TranslationDataset(
    data=sample_data,
    tokenizer=t5_tokenizer,
    shift_right_func=t5_model._shift_right
)
sample_loader = create_dataloader(sample_trans_data)

sample_valid_trans_data = TranslationDataset(
    data=sample_valid,
    tokenizer=t5_tokenizer,
    shift_right_func=t5_model._shift_right
)
sample_valid_loader = create_dataloader(sample_valid_trans_data)

In [12]:
training_params = TrainingHyperParams(
    num_epochs=1, 
    batch_size=16, 
    learning_rate=0.001, 
    warmup_steps=100, 
    max_grad_norm=1,
)

trainer = Trainer(
    model=prefixed_t5_model,
    tokenizer=t5_tokenizer,
    train_loader=train_loader,
    valid_loader=valid_loader,
    test_loader=test_loader,
    args=training_params
)

In [13]:
trainer.model

PrefixT5(
  (t5): T5ForConditionalGeneration(
    (shared): Embedding(32128, 512)
    (encoder): T5Stack(
      (embed_tokens): Embedding(32128, 512)
      (block): ModuleList(
        (0): T5Block(
          (layer): ModuleList(
            (0): T5LayerSelfAttention(
              (SelfAttention): T5Attention(
                (q): Linear(in_features=512, out_features=512, bias=False)
                (k): Linear(in_features=512, out_features=512, bias=False)
                (v): Linear(in_features=512, out_features=512, bias=False)
                (o): Linear(in_features=512, out_features=512, bias=False)
                (relative_attention_bias): Embedding(32, 8)
              )
              (layer_norm): T5LayerNorm()
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (1): T5LayerFF(
              (DenseReluDense): T5DenseActDense(
                (wi): Linear(in_features=512, out_features=2048, bias=False)
                (wo): Linear(in_features=2048,

In [14]:
train_loss, val_loss = trainer.train()

Validation: 100%|██████████| 715/715 [01:06<00:00, 10.74batch/s]


Epoch 1/1
	Train Loss: 0.7276
	Validation Loss: 0.6119

Model saved at models/prefix_t5-num_epochs_1-batch_size_16-learning_rate_0.001-warmup_steps_100-max_grad_norm_1-device_cuda.pth


In [15]:
# saved_model = load_model(base_model=t5_model, save_path="models/prefix_t5-num_epochs_1-batch_size_16-learning_rate_0.001-warmup_steps_100-max_grad_norm_1-device_cpu.pth")
# saved_model 
saved_model = load_model(base_model=t5_model, save_path="models/prefix_t5-num_epochs_1-batch_size_16-learning_rate_0.001-warmup_steps_100-max_grad_norm_1-device_cuda.pth")
saved_model 

Model successfully loaded from models/prefix_t5-num_epochs_1-batch_size_16-learning_rate_0.001-warmup_steps_100-max_grad_norm_1-device_cuda.pth


/home/bina/Documents/qazaq_llm/qazaq_llm/.conda/lib/python3.11/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


PrefixT5(
  (t5): T5ForConditionalGeneration(
    (shared): Embedding(32128, 512)
    (encoder): T5Stack(
      (embed_tokens): Embedding(32128, 512)
      (block): ModuleList(
        (0): T5Block(
          (layer): ModuleList(
            (0): T5LayerSelfAttention(
              (SelfAttention): T5Attention(
                (q): Linear(in_features=512, out_features=512, bias=False)
                (k): Linear(in_features=512, out_features=512, bias=False)
                (v): Linear(in_features=512, out_features=512, bias=False)
                (o): Linear(in_features=512, out_features=512, bias=False)
                (relative_attention_bias): Embedding(32, 8)
              )
              (layer_norm): T5LayerNorm()
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (1): T5LayerFF(
              (DenseReluDense): T5DenseActDense(
                (wi): Linear(in_features=512, out_features=2048, bias=False)
                (wo): Linear(in_features=2048,

In [47]:
res = generate_translation(
    model=saved_model,
    tokenizer=t5_tokenizer, 
    input_text="Hello my name is Sabina",
    max_length=1000,
)
res

NameError: name 'saved_model' is not defined

### Test models with different hyperparameters

In [55]:
prefixed_t5_model = PrefixT5(
    t5_model=t5_model, prefix_length=5, prefix_projection_size=512
)
model_1 = load_model(base_model=t5_model, prefix_length=5,save_path="../../root/models/prefix_len_5-proj_size_512-lr_8e-05.pth")

res = generate_translation(
    model=model_1,
    tokenizer=t5_tokenizer, 
    input_text="translate English to French: Hello my name is Sabina",
    max_length=1000,
)
res

/workspace/prefix-tuning/trainer.py:221: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(save_path, map_location=device)


	size mismatch for encoder_prefix.prefix_projection: copying a param with shape torch.Size([5, 512]) from checkpoint, the shape in current model is torch.Size([10, 512]).
	size mismatch for decoder_prefix.prefix_projection: copying a param with shape torch.Size([5, 512]) from checkpoint, the shape in current model is torch.Size([10, 512]).
Model successfully loaded from ../../root/models/prefix_len_5-proj_size_512-lr_8e-05.pth


'– a – a – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – – –'

In [61]:
prefix_length = 5  # Match the value used during training
prefix_projection_size = 512  # Match the value used during training

model_1 = PrefixT5(
    t5_model=t5_model,
    prefix_length=prefix_length,
    prefix_projection_size=prefix_projection_size
)

checkpoint = torch.load("../../root/models/prefix_len_5-proj_size_512-lr_8e-05.pth")
model_1.load_state_dict(checkpoint["model_state_dict"])
print("Model successfully loaded.")
res = generate_translation(
    model=model_1,
    tokenizer=t5_tokenizer, 
    input_text="translate English to French: Translation task is working well for English-French dataset",
    max_length=1000,
)
res


/tmp/ipykernel_18998/2482547224.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("../../root/models/prefix_len_5-proj_size_512-lr_8e-05.pth")


Model successfully loaded.


'La tâche de traduction fonctionne bien pour les ensembles de données anglais-français.'